# Tutorial 4: Building a Simple Cognitive Model

In this tutorial we'll work through building a complete cognitive model from scratch. We'll use the number comparison task as our example—participants decide which of two numbers is larger. It's simple enough to model completely, but complex enough to demonstrate how all the ACT-R components work together.

## Table of Contents
1. [The Number Comparison Task](#task)
2. [Designing the Model Architecture](#design)
3. [Implementing the Model](#implementation)
4. [Running Experiments](#experiments)
5. [Analyzing Results](#analysis)
6. [Model Validation](#validation)
7. [Parameter Exploration](#parameters)
8. [Exercises](#exercises)

## Introduction

The number comparison task has been studied extensively in cognitive psychology, and for good reason—it reveals several interesting phenomena:

1. **Distance Effect**: Numbers that are closer together (8 vs 9) take longer to compare than distant ones (2 vs 9)
2. **Size Effect**: Larger numbers take slightly longer to compare than smaller ones
3. **Practice Effects**: Performance improves with repetition

These effects tell us something about how numerical knowledge is represented and accessed in memory. Our model will attempt to capture these patterns using ACT-R's declarative memory and production systems.

In [ ]:
import pyactr as actr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import time
from collections import defaultdict

np.random.seed(42)
sns.set_style("darkgrid")
sns.set_palette("husl")

print("Libraries loaded. Ready to build the model.")

## The Number Comparison Task <a id='task'></a>

The task is straightforward: two numbers are presented, and the participant indicates which is larger. We measure response time and accuracy.

Before building our model, we need some "human" data to compare against. The code below generates synthetic data that follows the empirical patterns found in the literature.

In [ ]:
def generate_human_data(n_trials=100):
    """Generate realistic human RT data for number comparison"""
    data = []
    
    for _ in range(n_trials):
        num1 = np.random.randint(1, 10)
        num2 = np.random.randint(1, 10)
        while num2 == num1:
            num2 = np.random.randint(1, 10)
        
        distance = abs(num1 - num2)
        size = (num1 + num2) / 2
        
        # Base RT around 400ms, with distance and size effects
        base_rt = 400
        distance_effect = 200 / distance
        size_effect = 10 * size
        noise = np.random.normal(0, 50)
        
        rt = max(200, base_rt + distance_effect + size_effect + noise)
        
        error_prob = 0.05 / distance
        correct = np.random.random() > error_prob
        
        data.append({
            'num1': num1,
            'num2': num2,
            'distance': distance,
            'size': size,
            'rt': rt,
            'correct': correct,
            'larger': max(num1, num2)
        })
    
    return pd.DataFrame(data)

human_data = generate_human_data(200)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(human_data['distance'], human_data['rt'], alpha=0.5)
axes[0].set_xlabel('Number Distance')
axes[0].set_ylabel('Response Time (ms)')
axes[0].set_title('Distance Effect in Human Data')

axes[1].scatter(human_data['size'], human_data['rt'], alpha=0.5)
axes[1].set_xlabel('Average Magnitude')
axes[1].set_ylabel('Response Time (ms)')
axes[1].set_title('Size Effect in Human Data')

accuracy_by_distance = human_data.groupby('distance')['correct'].mean()
axes[2].bar(accuracy_by_distance.index, accuracy_by_distance.values)
axes[2].set_xlabel('Number Distance')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy by Distance')
axes[2].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

print(f"Human data summary:")
print(f"Mean RT: {human_data['rt'].mean():.0f}ms")
print(f"Accuracy: {human_data['correct'].mean():.1%}")

## Designing the Model Architecture <a id='design'></a>

Before we start coding, we should think through what the model needs.

**Declarative Memory** should contain number magnitude knowledge and comparison facts that have been learned through experience.

**Productions** will handle the cognitive process: attend to the numbers, try to retrieve a comparison fact, use a backup strategy if retrieval fails, and execute the response.

The basic flow:
1. Encode the two numbers
2. Attempt to retrieve a comparison fact from memory
3. If retrieval fails, fall back to a counting or magnitude estimation strategy
4. Execute response

This mirrors what we think humans actually do—they use memory when they can, and computation when they must.

## Implementing the Model <a id='implementation'></a>

Now we can build the model.

In [ ]:
class NumberComparisonModel:
    def __init__(self, name="NumberComparison"):
        self.model = actr.ACTRModel()
        self.model.buffers["imaginal"] = actr.Buffer()
        
        self.setup_parameters()
        self.setup_chunks()
        self.setup_memory()
        self.setup_productions()
        
    def setup_parameters(self):
        self.model.model_parameters["subsymbolic"] = True
        self.model.model_parameters["retrieval_threshold"] = -1.5
        self.model.model_parameters["latency_factor"] = 0.3
        self.model.model_parameters["latency_exponent"] = 1.0
        self.model.model_parameters["instantaneous_noise"] = 0.25
        self.model.model_parameters["decay"] = 0.5
        self.model.model_parameters["production_compilation"] = False
        self.model.model_parameters["utility_noise"] = 0.5
        
        print("Model parameters configured")
        
    def setup_chunks(self):
        actr.chunktype("goal", ("state", "num1", "num2", "answer", "strategy"))
        actr.chunktype("comparison", ("smaller", "larger", "difference"))
        actr.chunktype("number_fact", ("number", "magnitude"))
        
        print("Chunk types defined")
        
    def setup_memory(self):
        dm = self.model.decmem
        
        # Add magnitude facts for numbers 1-9
        for i in range(1, 10):
            dm.add(actr.makechunk(typename="number_fact",
                                number=str(i),
                                magnitude=i))
        
        # Pre-populate with some common comparisons
        common_comparisons = [
            (1, 9, 8), (2, 8, 6), (3, 7, 4),
            (1, 2, 1), (8, 9, 1), (4, 5, 1)
        ]
        
        for small, large, diff in common_comparisons:
            chunk = actr.makechunk(typename="comparison",
                                 smaller=str(small),
                                 larger=str(large),
                                 difference=diff)
            dm.add(chunk)
            dm.add(chunk)  # Add twice to increase base activation
            
        print(f"Declarative memory populated with {len(list(dm))} chunks")
        
    def setup_productions(self):
        self.model.productionstring(name="start_comparison",
            string="""
            =g>
                isa      goal
                state    ready
                num1     =n1
                num2     =n2
                answer   None
            ==>
            =g>
                state    retrieving
                strategy retrieval
            +retrieval>
                isa      comparison
                smaller  =n1
                larger   =n2
            """)
        
        self.model.productionstring(name="try_reverse_retrieval",
            string="""
            =g>
                isa      goal
                state    ready
                num1     =n1
                num2     =n2
                answer   None
            ==>
            =g>
                state    retrieving
                strategy retrieval
            +retrieval>
                isa      comparison
                smaller  =n2
                larger   =n1
            """)
        
        self.model.productionstring(name="retrieved_first_larger",
            string="""
            =g>
                isa      goal
                state    retrieving
                num1     =n1
                num2     =n2
            =retrieval>
                isa      comparison
                smaller  =n2
                larger   =n1
            ==>
            =g>
                state    done
                answer   =n1
            """)
        
        self.model.productionstring(name="retrieved_second_larger",
            string="""
            =g>
                isa      goal
                state    retrieving
                num1     =n1
                num2     =n2
            =retrieval>
                isa      comparison
                smaller  =n1
                larger   =n2
            ==>
            =g>
                state    done
                answer   =n2
            """)
        
        self.model.productionstring(name="retrieval_failed_count",
            string="""
            =g>
                isa      goal
                state    retrieving
                num1     =n1
                num2     =n2
            ?retrieval>
                state    error
            ==>
            =g>
                state    counting
                strategy counting
            +retrieval>
                isa      number_fact
                number   =n1
            """)
        
        self.model.productionstring(name="got_first_magnitude",
            string="""
            =g>
                isa      goal
                state    counting
                num1     =n1
                num2     =n2
            =retrieval>
                isa      number_fact
                number   =n1
                magnitude =m1
            ==>
            =g>
                state    get_second
            +retrieval>
                isa      number_fact
                number   =n2
            """)
        
        self.model.productionstring(name="compare_magnitudes",
            string="""
            =g>
                isa      goal
                state    get_second
                num1     =n1
                num2     =n2
            =retrieval>
                isa      number_fact
                number   =n2
                magnitude =m2
            ==>
            =g>
                state    done
                answer   =n2
                strategy counting
            """)
        
        self.model.productionstring(name="first_larger_counting",
            string="""
            =g>
                isa      goal  
                state    get_second
                num1     =n1
                num2     =n2
            =retrieval>
                isa      number_fact
                number   =n2
                magnitude 1
            ==>
            =g>
                state    done
                answer   =n1
                strategy counting
            """)
        
        print(f"Created {len(self.model.productions)} production rules")
    
    def run_trial(self, num1, num2, max_time=5.0):
        """Run a single comparison trial"""
        self.model.goal.clear()
        if hasattr(self.model, 'imaginal'):
            self.model.imaginal.clear()
        
        self.model.goal.add(actr.makechunk(typename="goal",
                                         state="ready",
                                         num1=str(num1),
                                         num2=str(num2)))
        
        sim = self.model.simulation(gui=False, trace=False)
        start_time = sim.show_time()
        sim.run(max_time)
        end_time = sim.show_time()
        
        goal_chunk = list(self.model.goal)[0] if list(self.model.goal) else None
        
        if goal_chunk and hasattr(goal_chunk, 'answer'):
            answer_val = goal_chunk.answer
            if answer_val and str(answer_val) != 'None':
                try:
                    response = int(str(answer_val))
                except:
                    response = None
            else:
                response = None
            
            rt = (end_time - start_time) * 1000
            strategy = str(goal_chunk.strategy) if hasattr(goal_chunk, 'strategy') else 'unknown'
            correct = response == max(num1, num2) if response else False
        else:
            response = None
            rt = max_time * 1000
            strategy = 'timeout'
            correct = False
            
        return {
            'num1': num1,
            'num2': num2,
            'response': response,
            'rt': rt,
            'correct': correct,
            'strategy': strategy,
            'distance': abs(num1 - num2),
            'size': (num1 + num2) / 2
        }

model = NumberComparisonModel()
print("\nModel created successfully!")
print("\nTesting with a single trial:")
result = model.run_trial(3, 7)
print(f"Comparing 3 vs 7: Response={result['response']}, RT={result['rt']:.0f}ms, Strategy={result['strategy']}")

## Running Experiments <a id='experiments'></a>

Now we can run a full experiment and compare the model to our synthetic human data.

In [ ]:
def run_experiment(model, n_trials=50):
    """Run a complete experiment with the model"""
    results = []
    
    for trial in range(n_trials):
        num1 = np.random.randint(1, 10)
        num2 = np.random.randint(1, 10)
        while num2 == num1:
            num2 = np.random.randint(1, 10)
            
        result = model.run_trial(num1, num2)
        result['trial'] = trial + 1
        results.append(result)
        
        if (trial + 1) % 10 == 0:
            print(f"Completed {trial + 1}/{n_trials} trials")
    
    return pd.DataFrame(results)

print("Running model experiment...")
model_data = run_experiment(model, n_trials=100)

print("\nModel Performance Summary:")
print(f"Mean RT: {model_data['rt'].mean():.0f}ms")
print(f"Accuracy: {model_data['correct'].mean():.1%}")
print(f"\nStrategy use:")
print(model_data['strategy'].value_counts())

model_data_valid = model_data[model_data['strategy'] != 'timeout']

## Analyzing Results <a id='analysis'></a>

Does our model capture the key phenomena?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Distance effect - human
axes[0, 0].scatter(human_data['distance'], human_data['rt'], alpha=0.5, label='Human')
axes[0, 0].set_xlabel('Number Distance')
axes[0, 0].set_ylabel('Response Time (ms)')
axes[0, 0].set_title('Human: Distance Effect')
axes[0, 0].set_ylim(300, 800)

# Distance effect - model
axes[1, 0].scatter(model_data_valid['distance'], model_data_valid['rt'], 
                   alpha=0.5, color='orange', label='Model')
axes[1, 0].set_xlabel('Number Distance')
axes[1, 0].set_ylabel('Response Time (ms)')
axes[1, 0].set_title('Model: Distance Effect')
axes[1, 0].set_ylim(300, 800)

# Size effect - human
axes[0, 1].scatter(human_data['size'], human_data['rt'], alpha=0.5)
axes[0, 1].set_xlabel('Average Magnitude')
axes[0, 1].set_ylabel('Response Time (ms)')
axes[0, 1].set_title('Human: Size Effect')
axes[0, 1].set_ylim(300, 800)

# Size effect - model
axes[1, 1].scatter(model_data_valid['size'], model_data_valid['rt'], 
                   alpha=0.5, color='orange')
axes[1, 1].set_xlabel('Average Magnitude')
axes[1, 1].set_ylabel('Response Time (ms)')
axes[1, 1].set_title('Model: Size Effect')
axes[1, 1].set_ylim(300, 800)

# Mean RT by distance
human_rt_by_dist = human_data.groupby('distance')['rt'].mean()
model_rt_by_dist = model_data_valid.groupby('distance')['rt'].mean()

axes[0, 2].plot(human_rt_by_dist.index, human_rt_by_dist.values, 'o-', label='Human')
axes[0, 2].set_xlabel('Number Distance')
axes[0, 2].set_ylabel('Mean RT (ms)')
axes[0, 2].set_title('Human: Mean RT by Distance')
axes[0, 2].legend()

axes[1, 2].plot(model_rt_by_dist.index, model_rt_by_dist.values, 'o-', 
                color='orange', label='Model')
axes[1, 2].set_xlabel('Number Distance')
axes[1, 2].set_ylabel('Mean RT (ms)')
axes[1, 2].set_title('Model: Mean RT by Distance')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

print("\nQuantitative comparison:")
print(f"Distance effect correlation:")

corr = 0
if len(human_rt_by_dist) == len(model_rt_by_dist) and len(human_rt_by_dist) > 2:
    common_distances = set(human_rt_by_dist.index) & set(model_rt_by_dist.index)
    if len(common_distances) >= 3:
        human_vals = [human_rt_by_dist[d] for d in sorted(common_distances)]
        model_vals = [model_rt_by_dist[d] for d in sorted(common_distances)]
        corr, p_value = stats.pearsonr(human_vals, model_vals)
        print(f"  r = {corr:.3f}, p = {p_value:.3f}")
    else:
        print("  Not enough common distance values for correlation")
else:
    print("  Data sizes don't match for correlation")
    
print(f"\nModel fit: {'Good' if corr > 0.7 else 'Moderate' if corr > 0.3 else 'Needs improvement'}")

## Model Validation <a id='validation'></a>

Looking at strategy use helps us understand how the model is achieving its behavior.

In [ ]:
strategy_by_distance = model_data_valid.groupby(['distance', 'strategy']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

strategy_by_distance.plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_xlabel('Number Distance')
axes[0].set_ylabel('Number of Trials')
axes[0].set_title('Strategy Use by Distance')
axes[0].legend(title='Strategy')

rt_by_strategy = model_data_valid.groupby('strategy')['rt'].mean()
rt_by_strategy.plot(kind='bar', ax=axes[1])
axes[1].set_xlabel('Strategy')
axes[1].set_ylabel('Mean RT (ms)')
axes[1].set_title('Response Time by Strategy')

plt.tight_layout()
plt.show()

print("\nThe model uses retrieval more for common comparisons and")
print("falls back to counting for unfamiliar pairs. This mirrors")
print("human behavior: fast memory-based responses when possible,")
print("slower computation when necessary.")

# Simulate learning by adding successful trials to memory
print("\n\nSimulating learning effects...")

for _, trial in model_data_valid.iterrows():
    if trial['correct']:
        smaller = min(trial['num1'], trial['num2'])
        larger = max(trial['num1'], trial['num2'])
        model.model.decmem.add(actr.makechunk(
            typename="comparison",
            smaller=str(smaller),
            larger=str(larger),
            difference=larger-smaller
        ))

print("Running second block after learning...")
model_data_block2 = run_experiment(model, n_trials=50)

print("\nLearning effects:")
print(f"Block 1 mean RT: {model_data_valid['rt'].mean():.0f}ms")
block2_valid = model_data_block2[model_data_block2['strategy'] != 'timeout']
print(f"Block 2 mean RT: {block2_valid['rt'].mean():.0f}ms")
print(f"Improvement: {model_data_valid['rt'].mean() - block2_valid['rt'].mean():.0f}ms")

print("\nStrategy shift with practice:")
print("Block 1:", model_data_valid['strategy'].value_counts(normalize=True))
print("Block 2:", block2_valid['strategy'].value_counts(normalize=True))

## Parameter Exploration <a id='parameters'></a>

Understanding how parameters affect behavior is crucial. These parameters represent cognitive mechanisms, and exploring them helps us understand what drives the effects we see.

In [ ]:
def test_parameter(param_name, param_values, n_trials=30):
    """Test how a parameter affects model behavior"""
    results = []
    
    for value in param_values:
        test_model = NumberComparisonModel()
        test_model.model.model_parameters[param_name] = value
        
        data = run_experiment(test_model, n_trials=n_trials)
        valid_data = data[data['strategy'] != 'timeout']
        
        results.append({
            'parameter': param_name,
            'value': value,
            'mean_rt': valid_data['rt'].mean(),
            'accuracy': valid_data['correct'].mean(),
            'retrieval_rate': (valid_data['strategy'] == 'retrieval').mean()
        })
        
    return pd.DataFrame(results)

print("Testing retrieval threshold effects...")
threshold_results = test_parameter('retrieval_threshold', 
                                 [-3, -2, -1.5, -1, -0.5], 
                                 n_trials=40)

print("\nTesting activation noise effects...")
noise_results = test_parameter('instantaneous_noise', 
                             [0, 0.1, 0.25, 0.5, 1.0], 
                             n_trials=40)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].plot(threshold_results['value'], threshold_results['mean_rt'], 'o-')
axes[0, 0].set_xlabel('Retrieval Threshold')
axes[0, 0].set_ylabel('Mean RT (ms)')
axes[0, 0].set_title('Threshold Effect on RT')

axes[0, 1].plot(threshold_results['value'], threshold_results['accuracy'], 'o-')
axes[0, 1].set_xlabel('Retrieval Threshold')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Threshold Effect on Accuracy')

axes[0, 2].plot(threshold_results['value'], threshold_results['retrieval_rate'], 'o-')
axes[0, 2].set_xlabel('Retrieval Threshold')
axes[0, 2].set_ylabel('Retrieval Use Rate')
axes[0, 2].set_title('Threshold Effect on Strategy')

axes[1, 0].plot(noise_results['value'], noise_results['mean_rt'], 'o-', color='orange')
axes[1, 0].set_xlabel('Activation Noise')
axes[1, 0].set_ylabel('Mean RT (ms)')
axes[1, 0].set_title('Noise Effect on RT')

axes[1, 1].plot(noise_results['value'], noise_results['accuracy'], 'o-', color='orange')
axes[1, 1].set_xlabel('Activation Noise')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Noise Effect on Accuracy')

axes[1, 2].plot(noise_results['value'], noise_results['retrieval_rate'], 'o-', color='orange')
axes[1, 2].set_xlabel('Activation Noise')
axes[1, 2].set_ylabel('Retrieval Use Rate')
axes[1, 2].set_title('Noise Effect on Strategy')

plt.tight_layout()
plt.show()

print("\nLower threshold means more retrieval attempts, which is faster but can lead to more errors.")
print("Higher noise creates more variable behavior, with occasional slow responses.")
print("These parameters can model individual differences between participants.")

## Model Insights

What did we learn from building this model?

**Emergent Behavior**: The distance and size effects emerge naturally from basic cognitive mechanisms—memory retrieval for familiar comparisons, procedural strategies for unfamiliar pairs, and activation dynamics affecting retrieval speed. We didn't explicitly program these effects; they arose from the architecture.

**Strategy Selection**: The model naturally develops efficient behavior by using fast retrieval when possible and falling back to slower computation when needed. This adaptive strategy selection mirrors what we see in human cognition.

**Individual Differences**: The parameters let us model different cognitive profiles. Retrieval threshold might correspond to working memory capacity, noise to processing variability, and decay to learning rate. By varying these, we could simulate different participants.

**Validation Matters**: Comparing model output to human data ensures psychological plausibility. A model that fits the data tells us our theory of the underlying mechanisms might be correct.

## Exercises <a id='exercises'></a>

### Exercise 1: Add Visual Encoding

Extend the model to include visual encoding time. Numbers presented visually should take time to encode before comparison begins. Does this improve the fit to human data?

In [ ]:
# Your code here
# Add a visual buffer and create productions for encoding
# Encoding time might depend on number magnitude

### Exercise 2: Model the Stroop Task

Build a complete model of the Stroop task where participants name the color of words that spell color names (e.g., "RED" printed in blue). You'll need visual processing for both color and word, competing productions for reading vs color naming, and conflict resolution mechanisms.

In [ ]:
# Your code here

### Exercise 3: Add Confidence Judgments

Extend the number comparison model to also predict confidence ratings. Confidence could relate to retrieval activation, the distance between numbers, or the strategy used. Compare model confidence to human confidence ratings if you have that data.

In [ ]:
# Your code here

### Exercise 4: Model Individual Differences

Create multiple model "participants" with different parameters and see if they match the variability in human data. Run each through the same experiment and analyze individual differences in overall speed, strategy preferences, and learning rates.

In [ ]:
# Your code here

## Summary

We've built a complete cognitive model that integrates memory, production rules, and learning. The model produces human-like behavior—distance effects, size effects, strategy selection—and these patterns emerge from basic cognitive mechanisms rather than being explicitly programmed.

A few things to remember when building your own models:

Start with the phenomena you're trying to explain. Design the architecture before you code. Validate quantitatively by comparing to real data. Remember that parameters represent cognitive mechanisms and individual differences.

Tutorial 5 covers advanced subsymbolic processing: memory dynamics, forgetting, spreading activation, production compilation, and modeling expertise development. These are the tools you'll need for modeling more complex behavior and learning over longer timescales.

---

## Additional Resources

Some relevant papers on number comparison and ACT-R modeling:
- Moyer, R. S., & Landauer, T. K. (1967). Time required for judgements of numerical inequality.
- Anderson, J. R. (2007). How can the human mind occur in the physical universe?
- Lebiere, C., & Anderson, J. R. (1993). A connectionist implementation of the ACT-R production system.

Model repositories worth exploring:
- ACT-R model library contains many complete models
- jACT-R is a Java implementation with additional examples
- ccm-suite is a Python cognitive modeling suite

If you want to apply this to your own work, try modeling a task from your research domain. Start simple—one or two key phenomena—and expand from there. The goal is insight into cognitive processes, not just data fitting.